# NLP-sentiment-analysis-twitter

In [1]:
import pandas as pd
import numpy as np
import re
import string

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# ML
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

# Evaluation
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

C:\Users\sarth\AppData\Local\Temp\ipykernel_29516\3356821837.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [5]:
import os

In [6]:
os.getcwd()

'C:\\Users\\sarth'

In [11]:
os.chdir(r"C:\Users\sarth\Downloads\archive (16)")

In [14]:
df=pd.read_csv("Twitter_Data.csv")

In [15]:
print(df.columns)
df.info()

Index(['clean_text', 'category'], dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 162980 entries, 0 to 162979
Data columns (total 2 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   clean_text  162976 non-null  object 
 1   category    162973 non-null  float64
dtypes: float64(1), object(1)
memory usage: 2.5+ MB


In [16]:
df.rename(columns={
    'clean_text': 'text',
    'category': 'sentiment'
}, inplace=True)

In [17]:
df.dropna(inplace=True)

In [18]:
def convert_sentiment(x):
    if x == -1:
        return "negative"
    elif x == 0:
        return "neutral"
    else:
        return "positive"

df['sentiment'] = df['sentiment'].apply(convert_sentiment)

In [19]:
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

nltk.download('stopwords')

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess(text):
    text = str(text).lower()
    
    # remove urls
    text = re.sub(r"http\S+", "", text)
    
    # remove punctuation
    text = re.sub(f"[{string.punctuation}]", "", text)
    
    # remove numbers
    text = re.sub(r"\d+", "", text)
    
    words = text.split()
    
    # remove stopwords
    words = [word for word in words if word not in stop_words]
    
    # stemming
    words = [stemmer.stem(word) for word in words]
    
    return " ".join(words)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\sarth\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [20]:
df['clean_text'] = df['text'].apply(preprocess)

In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)
X = tfidf.fit_transform(df['clean_text'])

In [22]:
from sklearn.model_selection import train_test_split

y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [23]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

C:\Users\sarth\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [24]:
from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)

In [25]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

In [26]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate(name, y_test, y_pred):
    print(f"\n{name}")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred, average='weighted'))
    print("Recall:", recall_score(y_test, y_pred, average='weighted'))
    print("F1 Score:", f1_score(y_test, y_pred, average='weighted'))

evaluate("Logistic Regression", y_test, y_pred_lr)
evaluate("Naive Bayes", y_test, y_pred_nb)
evaluate("Decision Tree", y_test, y_pred_dt)


Logistic Regression
Accuracy: 0.8423022642204087
Precision: 0.8432620134791842
Recall: 0.8423022642204087
F1 Score: 0.840975932818105

Naive Bayes
Accuracy: 0.6860158311345647
Precision: 0.7230251256015724
Recall: 0.6860158311345647
F1 Score: 0.6709393747763588

Decision Tree
Accuracy: 0.7713689636129349
Precision: 0.7700279873698875
Recall: 0.7713689636129349
F1 Score: 0.7705755226759434


In [27]:
import pandas as pd

results = pd.DataFrame({
    "Model": ["Logistic Regression", "Naive Bayes", "Decision Tree"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_nb),
        accuracy_score(y_test, y_pred_dt)
    ]
})

results

,Model,Accuracy
0,Logistic Regression,0.842302
1,Naive Bayes,0.686016
2,Decision Tree,0.771369


# - TF-IDF performed better than Bag of Words for Twitter data.
# - Logistic Regression gave highest accuracy among models.
# - Preprocessing improved model performance significantly.
# - Naive Bayes is fast but less accurate.
# - Decision Tree showed overfitting tendency.